### Build Constructors Dimension

1. Read silver constructors table
2. Read gold ref_nationality_region table
3. Join the data from constructors with ref_nationality_region using nationality
4. Select the required columns:
    * constructors.constructor_id
    * constructors.constructor_name
    * constructors.nationality
    * ref_nationality_region.region
5. Write the transformed data to gold dim_constructors table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/04.Gold-Helper

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table =F"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {target_table}")

In [0]:
constructors_df = (
    spark.read.table(F"{catalog_name}.{silver_schema}.constructors")
              .filter(F.col("batch_id") == v_batch_id)
)


ref_nationality_region_df = (
    spark.read.table(F"{catalog_name}.{gold_schema}.ref_nationality_region")
              
)



In [0]:
display(constructors_df)

In [0]:
dim_constructors_df =(
    constructors_df
        .join(ref_nationality_region_df, 
                constructors_df.nationality == ref_nationality_region_df.nationality, 
                "left")
        .select(constructors_df.constructor_id, 
                constructors_df.constructor_name, 
                constructors_df.nationality, 
                ref_nationality_region_df.region.alias("nationality_region") 
        )
)         
    

In [0]:
display(dim_constructors_df)

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region",
        
    ]

)

In [0]:
display(spark.table(target_table))